# EasyMagpieTTS via `vllm serve` — single streaming request

Send one TTS request to the OpenAI-compatible `vllm serve` endpoint
(`POST /v1/audio/speech`) and play the result.

Start the server first (in the `emp24` env, which has vLLM/vLLM-Omni 0.24):

```bash
./run_thin_server.sh /path/to/easymp_vllm_model_2stage 8091
```

We request **raw streaming audio** (`stream=true`, `stream_format="audio"`,
`response_format="pcm"`), so the server streams 16-bit PCM (`pcm_s16le`, mono
22050 Hz) chunk-by-chunk as the codec decodes each window. We measure
**time-to-first-audio (TTFA)** as the arrival of the first non-empty byte chunk.

Per-request knobs map onto the OpenAI schema:
- `input`  → text to synthesize
- `voice`  → EasyMagpie `speaker_id` (default `"eng"`)
- `extra_params` → optional `temperature` / `top_k` / `context_text` for the
  local-transformer sampling (forwarded via `additional_information`).

In [ ]:
import time

import numpy as np
import requests
import matplotlib.pyplot as plt
from IPython.display import Audio, display

SERVER_URL = "http://localhost:8091"
SPEECH_ENDPOINT = f"{SERVER_URL}/v1/audio/speech"
SAMPLE_RATE = 22050  # EasyMagpie codec output; raw PCM carries no header.

# Sanity check that the server is up and the engine finished loading.
# vLLM's /health returns 200 with an empty body (no JSON).
print("health:", requests.get(f"{SERVER_URL}/health", timeout=5).status_code)

In [ ]:
def synthesize(
    text: str,
    speaker_id: str = "eng",
    temperature: float | None = None,
    top_k: int | None = None,
    context_text: str | None = None,
    timeout: float = 300.0,
) -> tuple[np.ndarray, int]:
    """Stream raw PCM audio from `POST /v1/audio/speech` and collect it.

    Requests `stream_format="audio"` + `response_format="pcm"`, so the body is a
    continuous stream of 16-bit little-endian PCM (mono, 22050 Hz) delivered
    chunk-by-chunk as the codec decodes each window.
    """
    extra_params: dict = {}
    if temperature is not None:
        extra_params["temperature"] = temperature
    if top_k is not None:
        extra_params["top_k"] = top_k
    if context_text is not None:
        extra_params["context_text"] = context_text

    payload = {
        "input": text,
        "voice": speaker_id,
        "response_format": "pcm",
        "stream": True,
        "stream_format": "audio",
    }
    if extra_params:
        payload["extra_params"] = extra_params

    buf = bytearray()
    n_chunks = 0
    t0 = time.perf_counter()
    t_first = None

    with requests.post(SPEECH_ENDPOINT, json=payload, stream=True, timeout=timeout) as resp:
        resp.raise_for_status()
        for chunk in resp.iter_content(chunk_size=None):
            if not chunk:
                continue
            if t_first is None:
                t_first = time.perf_counter()
            buf.extend(chunk)
            n_chunks += 1

    elapsed = time.perf_counter() - t0
    ttfa = (t_first - t0) if t_first is not None else elapsed
    # pcm_s16le -> float32 in [-1, 1]. Drop a trailing odd byte if a chunk split a sample.
    if len(buf) % 2:
        buf = buf[:-1]
    audio = np.frombuffer(bytes(buf), dtype=np.int16).astype(np.float32) / 32768.0
    print(f"{n_chunks} chunks | TTFA: {ttfa * 1000:.0f}ms | total: {elapsed:.3f}s")
    return audio, SAMPLE_RATE

In [ ]:
audio, sr = synthesize(
    text="Since then physicists have found that it is not reflection, but refraction by the raindrops which causes the rainbows.",
)
print(f"Got {len(audio)} samples — {len(audio) / sr:.2f}s @ {sr} Hz")

In [ ]:
t = np.arange(len(audio)) / sr

fig, ax = plt.subplots(figsize=(12, 3))
ax.plot(t, audio, linewidth=0.3)
ax.set_xlabel("Time (s)")
ax.set_ylabel("Amplitude")
ax.set_title("EasyMagpieTTS waveform")
fig.tight_layout()
plt.show()

display(Audio(audio, rate=sr))